# 13 — ChemBERTa-2 Pretrained SMILES Embeddings

Uses `seyonec/ChemBERTa-zinc-base-v1` — a RoBERTa model pretrained on 77 million ZINC SMILES.
Frozen CLS-token embeddings (768-dim) are concatenated with Morgan FP as LGBM features.

**Hypothesis**: Pretraining on 77M diverse molecules encodes structural patterns
(aromaticity, ring strain, H-bond geometry) that ECFP4 and RDKit descriptors miss.

**Strategy**: Frozen embeddings (no fine-tuning on CPU) + LGBM regression head.
Fine-tuning a transformer on 3,781 examples would likely overfit on CPU; use as fixed encoder.

**Runtime**: ~30–60 min (CPU inference for 4,294 SMILES + LGBM CV).

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────────
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import lightgbm as lgb
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

from pxr.data import load_train, load_test
from pxr.chem import bemis_murcko
from pxr.featurize import morgan, impute
from pxr.eval import scaffold_kfold_indices, compute_metrics, rae as rae_fn
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

plt.rcParams.update({"figure.dpi": 120})
DEVICE = 'cpu'
MODEL_NAME = 'seyonec/ChemBERTa-zinc-base-v1'
print(f"torch {torch.__version__}  |  device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# ── 2. Load data ──────────────────────────────────────────────────────────────
train = load_train()
te    = load_test()

smiles_tr = train['smiles'].tolist()
smiles_te = te['smiles'].tolist()
y_tr      = train['pec50'].values

scaffolds = train['smiles'].map(bemis_murcko).tolist()
splits    = scaffold_kfold_indices(scaffolds, n_splits=5, seed=42)

print(f"Train: {len(smiles_tr):,}  |  Test: {len(smiles_te):,}")

In [ ]:
# ── 3. Load ChemBERTa-2 and extract embeddings ────────────────────────────────
cache_path_tr = DATA_PROCESSED / 'chemberta_train_emb.npy'
cache_path_te = DATA_PROCESSED / 'chemberta_test_emb.npy'

if cache_path_tr.exists() and cache_path_te.exists():
    print("Loading cached ChemBERTa embeddings ...")
    emb_tr = np.load(cache_path_tr)
    emb_te = np.load(cache_path_te)
else:
    print(f"Loading model from HuggingFace: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModel.from_pretrained(MODEL_NAME)
    model.eval()

    def encode_smiles(smiles_list: list[str], batch_size: int = 32) -> np.ndarray:
        """Extract CLS-token embeddings (frozen) for a list of SMILES."""
        all_embs = []
        for i in tqdm(range(0, len(smiles_list), batch_size), desc='Encoding'):
            batch = smiles_list[i : i + batch_size]
            enc   = tokenizer(batch, padding=True, truncation=True,
                               max_length=128, return_tensors='pt')
            with torch.no_grad():
                out = model(**enc)
            # CLS token = first token of last hidden state
            cls = out.last_hidden_state[:, 0, :].numpy()  # (batch, 768)
            all_embs.append(cls)
        return np.vstack(all_embs)

    print("Encoding training set ...")
    emb_tr = encode_smiles(smiles_tr)
    print("Encoding test set ...")
    emb_te = encode_smiles(smiles_te)

    np.save(cache_path_tr, emb_tr)
    np.save(cache_path_te, emb_te)
    print("Cached to data/processed/")

print(f"Embeddings: train {emb_tr.shape}  test {emb_te.shape}")

In [ ]:
# ── 4. Build feature matrix: ChemBERTa + Morgan FP ───────────────────────────
X_morgan_tr = morgan(smiles_tr).astype(np.float32)
X_morgan_te = morgan(smiles_te).astype(np.float32)

X_tr = np.hstack([emb_tr, X_morgan_tr])   # (N, 768 + 2048)
X_te = np.hstack([emb_te, X_morgan_te])
print(f"Combined feature matrix: {X_tr.shape}")

In [ ]:
# ── 5. Scaffold 5-fold CV with LGBM ──────────────────────────────────────────
lgbm_params = dict(
    n_estimators=1000, num_leaves=64, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=0.1,
    min_child_samples=10, n_jobs=4, verbose=-1,
)

oof_preds = np.full(len(y_tr), np.nan)
fold_metrics = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    m = lgb.LGBMRegressor(**lgbm_params)
    m.fit(X_tr[tr_idx], y_tr[tr_idx])
    oof_preds[va_idx] = m.predict(X_tr[va_idx])
    met = compute_metrics(y_tr[va_idx], oof_preds[va_idx])
    met['fold'] = fold
    fold_metrics.append(met)
    print(f"  Fold {fold+1}: RAE={met['RAE']:.4f}  Spearman={met['Spearman']:.4f}")

oof_rae = rae_fn(y_tr, oof_preds)
cv_df   = pd.DataFrame(fold_metrics)
print(f"\nOOF RAE (global): {oof_rae:.4f}")
print(f"Mean fold RAE:    {cv_df['RAE'].mean():.4f} +/- {cv_df['RAE'].std():.4f}")
print()
print("== Comparison ==")
print(f"  LGBM_base (Morgan + RDKit only): ~0.575")
print(f"  LGBM_aug (+ null feature):       0.5582")
print(f"  ChemBERTa + Morgan (this nb):    {oof_rae:.4f}")

In [ ]:
# ── 6. Train final model + predict test ──────────────────────────────────────
final_model = lgb.LGBMRegressor(**lgbm_params)
final_model.fit(X_tr, y_tr)
chemberta_preds = final_model.predict(X_te)
chemberta_preds = np.clip(chemberta_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)

# Blend with Chemprop using inverse-RAE weights
for fname in ['10_expanded_multitask.csv','08_chemprop_cv_blend.csv']:
    cp_path = SUBMISSIONS / fname
    if cp_path.exists():
        cp_preds = pd.read_csv(cp_path).set_index('Molecule Name').loc[te['name'].values,'pEC50'].values
        break

chemprop_rae  = 0.5736
w_cb = (1/oof_rae) / (1/oof_rae + 1/chemprop_rae)
final_preds = w_cb * chemberta_preds + (1 - w_cb) * cp_preds
final_preds = np.clip(final_preds, y_tr.min() - 0.5, y_tr.max() + 0.5)

print(f"ChemBERTa test preds: {chemberta_preds.min():.2f}–{chemberta_preds.max():.2f}")
print(f"ChemBERTa weight: {w_cb:.3f}  (RAE {oof_rae:.4f} vs Chemprop {chemprop_rae:.4f})")

In [ ]:
# ── 7. Save submission ────────────────────────────────────────────────────────
sub = pd.DataFrame({'Molecule Name': te['name'].values,
                    'SMILES':        te['smiles'].values,
                    'pEC50':         final_preds})
assert len(sub) == 513 and sub['pEC50'].notna().all()
out = SUBMISSIONS / '13_chemberta.csv'
sub.to_csv(out, index=False)

np.save(DATA_PROCESSED / 'oof_chemberta.npy', oof_preds)
np.save(DATA_PROCESSED / 'te_chemberta.npy',  chemberta_preds)

print(f"Saved: {out}")
print(f"ChemBERTa OOF RAE: {oof_rae:.4f}  |  blend weight: {w_cb:.3f}")
print(sub['pEC50'].describe().round(3))